In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

import torchaudio.transforms as T
import torch
import torchaudio

In [2]:
SAMPLE_RATE = 16000
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MEL_MEAN = -4.0
MEL_STD = 4.0

In [3]:
_mel_transform = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    win_length=WIN_LENGTH,
    n_mels=N_MELS,
).to(DEVICE)

In [4]:
def wav_to_mel(path: str) -> torch.Tensor:
    wav, sr = torchaudio.load(path)
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
    wav = wav.mean(dim=0, keepdim=True).to(DEVICE)
    mel = _mel_transform(wav).squeeze(0)
    amp_to_db = T.AmplitudeToDB()
    mel = amp_to_db(mel)
    return mel

In [7]:
spec = wav_to_mel("/workspaces/super-duper-dollop/data/dataset/speaker_01/after/2.wav")
spec1 = wav_to_mel("/workspaces/super-duper-dollop/data/dataset/speaker_01/before/2.wav")


In [9]:
spec.shape

torch.Size([128, 280])

In [8]:
from src.visualizations import viz_utils


viz_utils.save_spectrogram(spec, spec1, "spec.png",sample_rate=SAMPLE_RATE, hop_length=HOP_LENGTH)